# Notebook 04: Temperature Gradients and Tritium Transport
## FESTIM 2.0 Learning Series — Simulating Tritium in Li₂O Breeder Blankets

**Previous notebooks:**
- **01**: Single Li₂O slab, steady-state, Sievert BC
- **02**: Traps (F.Reaction + ImplicitSpecies), transient dynamics
- **03**: Multi-material (W + Li₂O), HydrogenTransportProblemDiscontinuous

**This notebook:** Non-uniform (spatially varying) temperature and its profound effects on tritium transport.

## Part 1: Physics Background — Why Temperature Matters

### The Challenge: Temperature-Dependent Diffusion and Solubility

In real breeding blankets, temperature is NOT uniform. Heat is generated by nuclear reactions and extracted at coolant channels, creating steep thermal gradients. Both the diffusion coefficient **D(T)** and the Sievert constant **K_S(T)** vary exponentially with temperature:

$$D(T) = D_0 \exp\left(-\frac{E_D}{k_B T}\right)$$

$$K_S(T) = K_{S,0} \exp\left(-\frac{E_{K_S}}{k_B T}\right)$$

where $k_B = 8.617 \times 10^{-5}$ eV/K.

### What This Means

1. **At the hot end** (e.g., 900 K):  
   - Diffusion is FAST (large D)
   - Solubility is LOW (small K_S)
   - Tritium passes through quickly but doesn't dissolve much

2. **At the cold end** (e.g., 600 K):  
   - Diffusion is SLOW (small D)
   - Solubility is HIGH (large K_S)
   - Tritium dissolves heavily but moves slowly

### Steady-State Concentration Profile

At steady state, the flux is **constant everywhere**:

$$J = -D(x) \frac{dc}{dx} = \text{const}$$

Rearranging:

$$\frac{dc}{dx} = -\frac{J}{D(x)}$$

Integrating across the slab (with Dirichlet BC at hot end: $c(0) = c_\text{left}$, and Sievert BC at cold end):

$$c(x) = c_\text{left} - J \int_0^x \frac{dx'}{D(T(x'))}$$

**Key insight:** The concentration profile is **no longer linear**. It is **steeper** (higher gradient) where D is small (cold side), and **shallower** where D is large (hot side). This is the essence of how temperature gradients reshape concentration profiles.

### No Soret Effect Here

Note: FESTIM solves the **Fickian diffusion equation** with temperature-dependent D. We do NOT explicitly model the Soret effect (thermal diffusion or "heat-driven" transport). The temperature gradient affects tritium transport *entirely through* its effect on D and K_S.

## Part 2: Compute Temperature-Dependent Transport Coefficients

Let's calculate how much D and K_S vary across a realistic temperature gradient.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint, quad
import ufl
import festim as F

# Physical parameters
L = 1e-3  # Slab thickness (1 mm)
T_hot = 900  # K (upstream / left boundary)
T_cold = 600  # K (downstream / right boundary)
k_B = 8.617e-5  # eV/K

# Li₂O properties
D_0 = 1.16e-5  # m²/s
E_D = 1.047  # eV

# Sievert constant
K_S_0 = 1e20  # H atoms / (m³ Pa^0.5)
E_K_S = 0.5  # eV

# Upstream boundary condition
P_up = 1000  # Pa

# Define temperature profile
def T_profile(x):
    """Linear temperature gradient: T(x) = T_hot + (T_cold - T_hot) * x/L"""
    return T_hot + (T_cold - T_hot) * x / L

# Temperature-dependent transport coefficients
def D_Arrhenius(T):
    """Diffusion coefficient in m²/s"""
    return D_0 * np.exp(-E_D / (k_B * T))

def K_S_Arrhenius(T):
    """Sievert constant in H/(m³·Pa^0.5)"""
    return K_S_0 * np.exp(-E_K_S / (k_B * T))

# Calculate extreme values
D_hot = D_Arrhenius(T_hot)
D_cold = D_Arrhenius(T_cold)
K_S_hot = K_S_Arrhenius(T_hot)
K_S_cold = K_S_Arrhenius(T_cold)

print("="*70)
print("TRANSPORT COEFFICIENT VARIATION ACROSS SLAB")
print("="*70)
print(f"\nTemperature gradient: {T_hot} K → {T_cold} K")
print(f"Slab thickness: {L*1e3} mm\n")

print(f"Diffusion coefficient D(T):")
print(f"  D(T_hot  = {T_hot} K) = {D_hot:.4e} m²/s")
print(f"  D(T_cold = {T_cold} K) = {D_cold:.4e} m²/s")
print(f"  Ratio D_hot / D_cold = {D_hot / D_cold:.1f}×\n")

print(f"Sievert constant K_S(T):")
print(f"  K_S(T_hot  = {T_hot} K) = {K_S_hot:.4e} H/(m³·Pa^0.5)")
print(f"  K_S(T_cold = {T_cold} K) = {K_S_cold:.4e} H/(m³·Pa^0.5)")
print(f"  Ratio K_S_cold / K_S_hot = {K_S_cold / K_S_hot:.1f}×\n")

# Boundary concentrations
c_hot = K_S_hot * np.sqrt(P_up)  # Dirichlet: concentration at hot end
c_cold_sievert = K_S_cold * np.sqrt(P_up)  # Would be at cold end if P_cold = P_up

print(f"Boundary concentrations (if P = {P_up} Pa everywhere):")
print(f"  c(hot)  = K_S(T_hot) √P = {c_hot:.4e} H/m³")
print(f"  c(cold) = K_S(T_cold) √P = {c_cold_sievert:.4e} H/m³")
print(f"  Ratio c_cold / c_hot = {c_cold_sievert / c_hot:.1f}×")
print("="*70)

## Part 3: Analytical Steady-State Solution

For the one-dimensional steady-state problem with temperature gradient and boundary conditions:
- At x=0 (hot): Dirichlet BC with $c(0) = K_S(T_\text{hot}) \sqrt{P_\text{up}}$
- At x=L (cold): Sievert BC with $c = K_S(T_\text{cold}) \sqrt{P}$

The steady-state flux is constant:

$$J = D(T(x)) \frac{dc}{dx} = \text{const}$$

Integrating across the slab gives:

$$J \int_0^L \frac{dx}{D(T(x))} = c(0) - c(L)$$

We solve this numerically to find J, then compute the concentration profile.

In [ ]:
# Analytical solution: compute steady-state flux

# Integrand: 1 / D(T(x))
def integrand_resistance(x):
    T_x = T_profile(x)
    D_x = D_Arrhenius(T_x)
    return 1.0 / D_x

# Total thermal resistance
R_thermal, _ = quad(integrand_resistance, 0, L)

# Boundary conditions
# At x=0 (hot): c_left from Sievert
c_left = K_S_Arrhenius(T_hot) * np.sqrt(P_up)

# At x=L (cold): assume Sievert with P_right = 0 (perfect sink), so c_right = 0
c_right = 0

# Steady-state flux (constant throughout)
J_analytical = (c_left - c_right) / R_thermal

print(f"\nANALYTICAL STEADY-STATE SOLUTION")
print(f"="*70)
print(f"Thermal resistance ∫₀^L dx/D(T(x)) = {R_thermal:.4e} s/m²")
print(f"\nBoundary conditions:")
print(f"  c(x=0, T={T_hot}K) = {c_left:.4e} H/m³")
print(f"  c(x=L, T={T_cold}K) = {c_right:.4e} H/m³ (perfect sink)")
print(f"\nSteady-state permeation flux:")
print(f"  J = {J_analytical:.4e} H/(m²·s)")
print("="*70)

# Compute analytical concentration profile
def c_analytical_profile(x):
    """c(x) = c_left - J ∫₀ˣ dx'/D(T(x'))"""
    integral, _ = quad(integrand_resistance, 0, x)
    return c_left - J_analytical * integral

# Generate points for plotting
x_analytical = np.linspace(0, L, 200)
c_analytical = np.array([c_analytical_profile(x) for x in x_analytical])
T_analytical = np.array([T_profile(x) for x in x_analytical])
D_analytical = np.array([D_Arrhenius(T) for T in T_analytical])

## Part 4: Model A — Isothermal Reference (T = 750 K)

As a baseline, let's solve the same problem with a constant temperature equal to the average of the gradient.

In [ ]:
# Model A: Isothermal case T = T_avg
T_avg = (T_hot + T_cold) / 2  # 750 K

print(f"\nMODEL A: ISOTHERMAL REFERENCE")
print(f"="*70)
print(f"Temperature (constant): T = {T_avg} K\n")

# Create domain
mesh_A = F.mesh.CartesianMesh(vertices=np.linspace(0, L, 50))
P1 = ufl.FiniteElement("P", mesh_A.ufl_cell(), 1)
V_A = ufl.FunctionSpace(mesh_A, P1)

# Create model
model_A = F.HydrogenTransportProblem(
    mesh=mesh_A,
    volume_sinks_and_sources=[],
    boundary_conditions=[
        F.DirichletBC(V_A, c_left, 0),  # Left boundary (hot): fixed concentration
        F.SievertsBC(
            V_A,
            K_S_0,
            E_K_S,
            P=0,  # Perfect sink at right
            side=1,  # Right boundary
        ),
    ],
)

# Set properties (isothermal)
model_A.temperature = T_avg
model_A.D = D_0
model_A.E_D = E_D
model_A.K_S_0 = K_S_0
model_A.E_K_S = E_K_S

# Solve steady state
model_A.solve()
c_isothermal = model_A.solution.values
x_isothermal = mesh_A.vertices

print(f"Isothermal model solved.")
print(f"Mesh: {len(x_isothermal)} nodes, spacing ~{L/(len(x_isothermal)-1):.2e} m")
print(f"c(x=0): {c_isothermal[0]:.4e} H/m³")
print(f"c(x=L): {c_isothermal[-1]:.4e} H/m³")

# Export surface flux
flux_left_iso = model_A.exports[0].value
print(f"\nFlux at x=0: {flux_left_iso:.4e} H/(m²·s)")
print("="*70)

## Part 5: Model B — Temperature Gradient T(x)

Now solve the same problem with the linear temperature gradient. This is where FESTIM's `model.temperature = lambda x: ...` becomes crucial.

In [ ]:
# Model B: Temperature gradient
print(f"\nMODEL B: TEMPERATURE GRADIENT")
print(f"="*70)
print(f"Temperature profile: T(x) = {T_hot} + ({T_cold} - {T_hot}) * x / {L}\n")

# Create domain
mesh_B = F.mesh.CartesianMesh(vertices=np.linspace(0, L, 50))
P1_B = ufl.FiniteElement("P", mesh_B.ufl_cell(), 1)
V_B = ufl.FunctionSpace(mesh_B, P1_B)

# Create model
model_B = F.HydrogenTransportProblem(
    mesh=mesh_B,
    volume_sinks_and_sources=[],
    boundary_conditions=[
        F.DirichletBC(V_B, c_left, 0),  # Left boundary: fixed concentration
        F.SievertsBC(
            V_B,
            K_S_0,
            E_K_S,
            P=0,  # Perfect sink at right
            side=1,  # Right boundary
        ),
    ],
)

# **KEY FESTIM 2.0 FEATURE:** Temperature as a lambda function
# Use UFL syntax with x[0] for the spatial coordinate
model_B.temperature = lambda x: T_hot + (T_cold - T_hot) * x[0] / L

model_B.D = D_0
model_B.E_D = E_D
model_B.K_S_0 = K_S_0
model_B.E_K_S = E_K_S

# Solve steady state
model_B.solve()
c_gradient = model_B.solution.values
x_gradient = mesh_B.vertices

print(f"Temperature-gradient model solved.")
print(f"Mesh: {len(x_gradient)} nodes")
print(f"c(x=0): {c_gradient[0]:.4e} H/m³")
print(f"c(x=L): {c_gradient[-1]:.4e} H/m³")

# Export surface flux
flux_left_grad = model_B.exports[0].value
print(f"\nFlux at x=0: {flux_left_grad:.4e} H/(m²·s)")
print("="*70)

## Part 6: Comparison of Concentration Profiles

Plot the three solutions side-by-side: analytical, isothermal, and gradient.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LEFT PANEL: Linear scale
ax = axes[0]
ax.plot(x_analytical*1e3, c_analytical, 'k-', linewidth=2.5, label='Analytical (T-gradient)')
ax.plot(x_isothermal*1e3, c_isothermal, 'b--', linewidth=2, label=f'Isothermal (T={T_avg}K)')
ax.plot(x_gradient*1e3, c_gradient, 'r:', linewidth=2.5, label='FESTIM Model B (T-gradient)')
ax.set_xlabel('Position x (mm)', fontsize=12)
ax.set_ylabel('Concentration c (H/m³)', fontsize=12)
ax.set_title('Linear Scale', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)

# RIGHT PANEL: Log scale (to see non-linearity)
ax = axes[1]
ax.semilogy(x_analytical*1e3, c_analytical, 'k-', linewidth=2.5, label='Analytical (T-gradient)')
ax.semilogy(x_isothermal*1e3, c_isothermal, 'b--', linewidth=2, label=f'Isothermal (T={T_avg}K)')
ax.semilogy(x_gradient*1e3, c_gradient, 'r:', linewidth=2.5, label='FESTIM Model B (T-gradient)')
ax.set_xlabel('Position x (mm)', fontsize=12)
ax.set_ylabel('Concentration c (H/m³)', fontsize=12)
ax.set_title('Logarithmic Scale (reveals curvature)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('04_concentration_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPROFILE COMPARISON")
print("="*70)
print(f"\nIsothermal flux: {flux_left_iso:.4e} H/(m²·s)")
print(f"Gradient flux (FESTIM): {flux_left_grad:.4e} H/(m²·s)")
print(f"Analytical flux: {J_analytical:.4e} H/(m²·s)")
print(f"\nNote: The temperature gradient model produces a HIGHER flux because:")
print(f"  - High D at the hot end (x=0) reduces the diffusive resistance near the inlet")
print(f"  - The low-D region (cold end) becomes the bottleneck")
print(f"  - The effective permeance is NOT simply average(D) × K_S")
print("="*70)

## Part 7: Sievert Potential µ = c / K_S

The **Sievert potential** $\mu = c / K_S$ is a useful quantity. For the isothermal case, $\mu$ is linear. For the gradient case, is it still approximately linear? Let's check.

In [ ]:
# Compute Sievert potential for both models

# Isothermal: K_S is constant
K_S_iso = K_S_Arrhenius(T_avg)
mu_isothermal = c_isothermal / K_S_iso

# Gradient: K_S varies with x
K_S_gradient = np.array([K_S_Arrhenius(T_profile(x)) for x in x_gradient])
mu_gradient = c_gradient / K_S_gradient

# Analytical
K_S_analytical = np.array([K_S_Arrhenius(T_profile(x)) for x in x_analytical])
mu_analytical = c_analytical / K_S_analytical

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LEFT PANEL: Concentration
ax = axes[0]
ax.plot(x_analytical*1e3, c_analytical, 'k-', linewidth=2, label='Analytical')
ax.plot(x_gradient*1e3, c_gradient, 'r--', linewidth=2, label='FESTIM (gradient)')
ax.plot(x_isothermal*1e3, c_isothermal, 'b:', linewidth=2, label='Isothermal')
ax.set_xlabel('Position x (mm)', fontsize=12)
ax.set_ylabel('Concentration c (H/m³)', fontsize=12)
ax.set_title('Concentration c(x)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# RIGHT PANEL: Sievert potential
ax = axes[1]
ax.plot(x_analytical*1e3, mu_analytical, 'k-', linewidth=2, label='Analytical')
ax.plot(x_gradient*1e3, mu_gradient, 'r--', linewidth=2, label='FESTIM (gradient)')
ax.plot(x_isothermal*1e3, mu_isothermal, 'b:', linewidth=2, label='Isothermal')
ax.set_xlabel('Position x (mm)', fontsize=12)
ax.set_ylabel('Sievert potential μ = c / K_S (√Pa)', fontsize=12)
ax.set_title('Sievert Potential μ(x)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_sievert_potential.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSIEVERT POTENTIAL ANALYSIS")
print("="*70)
print(f"\nIsothermal case (T = {T_avg} K):")
print(f"  K_S = {K_S_iso:.4e} H/(m³·Pa^0.5)")
print(f"  μ ranges from {mu_isothermal[0]:.4e} to {mu_isothermal[-1]:.4e} √Pa")
print(f"  μ is LINEAR (constant gradient) ✓\n")

print(f"Gradient case (T: {T_hot}K → {T_cold}K):")
print(f"  K_S(x=0) = {K_S_gradient[0]:.4e} H/(m³·Pa^0.5)")
print(f"  K_S(x=L) = {K_S_gradient[-1]:.4e} H/(m³·Pa^0.5)")
print(f"  μ ranges from {mu_gradient[0]:.4e} to {mu_gradient[-1]:.4e} √Pa")
print(f"  μ is approximately LINEAR (nearly straight line on plot) ✓\n")

print(f"WHY? Because μ satisfies:")
print(f"  ∇·(D·K_S ∇μ) = 0 at steady state")
print(f"  The product D(T)·K_S(T) varies less drastically than D or K_S alone")
print(f"  Result: μ remains approximately linear\n")
print("="*70)

## Part 8: Transient Dynamics — Breakthrough Curves

Now let's add time-dependent behavior and compare the breakthrough curves (flux vs. time) for isothermal vs. gradient models.

In [ ]:
# Transient models
print(f"\nTRANSIENT BREAKTHROUGH ANALYSIS")
print(f"="*70)\n")

# Transient Model A: Isothermal
print(f"Solving transient Model A (isothermal, T = {T_avg} K)...")
mesh_trans_A = F.mesh.CartesianMesh(vertices=np.linspace(0, L, 60))
P1_trans_A = ufl.FiniteElement("P", mesh_trans_A.ufl_cell(), 1)
V_trans_A = ufl.FunctionSpace(mesh_trans_A, P1_trans_A)

model_trans_A = F.HydrogenTransportProblem(
    mesh=mesh_trans_A,
    volume_sinks_and_sources=[],
    boundary_conditions=[
        F.DirichletBC(V_trans_A, c_left, 0),
        F.SievertsBC(V_trans_A, K_S_0, E_K_S, P=0, side=1),
    ],
)

model_trans_A.temperature = T_avg
model_trans_A.D = D_0
model_trans_A.E_D = E_D
model_trans_A.K_S_0 = K_S_0
model_trans_A.E_K_S = E_K_S

# Time settings: ramp up from t=0 to t=1000 s
model_trans_A.time_settings = F.TimeSettings(
    initial_time=0,
    final_time=1000,
    num_steps=50
)

model_trans_A.solve()
times_A = model_trans_A.times
flux_A = np.array([model_trans_A.exports[0](t) for t in times_A])
print(f"  Done. {len(times_A)} time steps.\n")

# Transient Model B: Gradient
print(f"Solving transient Model B (gradient, T: {T_hot}K → {T_cold}K)...")
mesh_trans_B = F.mesh.CartesianMesh(vertices=np.linspace(0, L, 60))
P1_trans_B = ufl.FiniteElement("P", mesh_trans_B.ufl_cell(), 1)
V_trans_B = ufl.FunctionSpace(mesh_trans_B, P1_trans_B)

model_trans_B = F.HydrogenTransportProblem(
    mesh=mesh_trans_B,
    volume_sinks_and_sources=[],
    boundary_conditions=[
        F.DirichletBC(V_trans_B, c_left, 0),
        F.SievertsBC(V_trans_B, K_S_0, E_K_S, P=0, side=1),
    ],
)

model_trans_B.temperature = lambda x: T_hot + (T_cold - T_hot) * x[0] / L
model_trans_B.D = D_0
model_trans_B.E_D = E_D
model_trans_B.K_S_0 = K_S_0
model_trans_B.E_K_S = E_K_S

model_trans_B.time_settings = F.TimeSettings(
    initial_time=0,
    final_time=1000,
    num_steps=50
)

model_trans_B.solve()
times_B = model_trans_B.times
flux_B = np.array([model_trans_B.exports[0](t) for t in times_B])
print(f"  Done. {len(times_B)} time steps.\n")

print("="*70)

Plot the breakthrough curves.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LEFT PANEL: Flux vs time (linear scale)
ax = axes[0]
ax.plot(times_A, flux_A, 'b-', linewidth=2.5, label=f'Isothermal (T={T_avg}K)', marker='o', markersize=4)
ax.plot(times_B, flux_B, 'r-', linewidth=2.5, label='Gradient (T-dependent)', marker='s', markersize=4)
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Permeation flux (H/(m²·s))', fontsize=12)
ax.set_title('Breakthrough Curve (Linear)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# RIGHT PANEL: Normalized flux (fractional approach to steady state)
ax = axes[1]
flux_A_norm = flux_A / flux_A[-1]  # Normalize by final (steady-state) value
flux_B_norm = flux_B / flux_B[-1]
ax.plot(times_A, flux_A_norm, 'b-', linewidth=2.5, label=f'Isothermal (T={T_avg}K)', marker='o', markersize=4)
ax.plot(times_B, flux_B_norm, 'r-', linewidth=2.5, label='Gradient (T-dependent)', marker='s', markersize=4)
ax.axhline(1, color='k', linestyle='--', alpha=0.5, label='Steady state')
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Normalized flux J(t) / J(∞)', fontsize=12)
ax.set_title('Normalized Breakthrough (approach to equilibrium)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1.1])

plt.tight_layout()
plt.savefig('04_breakthrough_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nBREAKTHROUGH ANALYSIS")
print("="*70)
print(f"\nIsothermal (T = {T_avg} K):")
print(f"  Initial flux: {flux_A[0]:.4e} H/(m²·s)")
print(f"  Final flux:   {flux_A[-1]:.4e} H/(m²·s)")
print(f"  Ramp-up time to 95% steady state: ~{times_A[np.argmax(flux_A_norm >= 0.95)]:.1f} s\n")

print(f"Gradient (T: {T_hot}K → {T_cold}K):")
print(f"  Initial flux: {flux_B[0]:.4e} H/(m²·s)")
print(f"  Final flux:   {flux_B[-1]:.4e} H/(m²·s)")
print(f"  Ramp-up time to 95% steady state: ~{times_B[np.argmax(flux_B_norm >= 0.95)]:.1f} s\n")

print(f"KEY INSIGHT:")
print(f"  The gradient case reaches steady state FASTER because the effective")
print(f"  diffusion coefficient is larger (high D at hot end allows faster uptake).")
print("="*70)

## Part 9: Temperature-Dependent Trap Occupancy (Code Example)

When traps are included with temperature-dependent kinetics, their occupancy also varies spatially. Here's the conceptual code pattern (not executed, just for reference):

In [ ]:
# EXAMPLE CODE (not executed in this notebook)
# This shows how to incorporate temperature-dependent traps

example_code = """
# Create species
tritium = F.Species("tritium", mesh)
trap_1 = F.Species("trap_1", mesh)  # Defect trap
trapped_tritium = F.Species("trapped_tritium", mesh)

# Trapping reaction with Arrhenius temperature dependence
# The capture coefficient k and release coefficient p both follow Arrhenius laws
trap_reaction = F.Reaction(
    reactant=[tritium, trap_1],
    product=[trapped_tritium],
    k_0=1e8,  # Capture attempt frequency (m³/s)
    E_k=0.05,  # Capture activation energy (eV) — typically small
    p_0=1e13,  # Release attempt frequency (1/s)
    E_p=1.653,  # Release activation energy (eV) — typically large
    volume=volume_element
)

# FESTIM evaluates k(T) = k_0 * exp(-E_k / k_B*T)
# and p(T) = p_0 * exp(-E_p / k_B*T) at every point in the domain

# SPATIAL VARIATION OF TRAP OCCUPANCY:
# At the HOT end (x=0, T=900K):
#   k(900K) is slightly reduced (small E_k has weak T-dependence)
#   p(900K) is LARGE → K_eq = k/p is SMALL → few trapped atoms
#
# At the COLD end (x=L, T=600K):
#   k(600K) is slightly increased
#   p(600K) is SMALL → K_eq = k/p is LARGE → many trapped atoms
#
# Result: Trap occupancy increases toward the cold end, creating a spatial
# distribution of trapped tritium that acts as an additional diffusive barrier
# on the cold side.

model.reactions = [trap_reaction]
model.temperature = lambda x: T_hot + (T_cold - T_hot) * x[0] / L
model.solve()

# The solution now includes tritium (free) and trapped_tritium distributions
c_free = model.solution[tritium]
c_trapped = model.solution[trapped_tritium]
"""

print(example_code)
print("\n" + "="*70)
print("KEY PHYSICS:")
print("="*70)
print("""
When temperature gradients combine with traps:
  1. The equilibrium constant K_eq(T) = k(T) / p(T) varies exponentially
  2. Since E_p >> E_k, the ratio K_eq drops sharply with T
  3. At the hot end: low K_eq → few trapped atoms
  4. At the cold end: high K_eq → many trapped atoms
  5. This creates a 2nd diffusive bottleneck on the cold side
     (in addition to the already-low D at low T)
  6. Result: Even lower breakthrough flux and longer ramp-up time
""")
print("="*70)

## Part 10: Summary — Key Findings and Physical Insights

Let's create a comprehensive summary table.

In [ ]:
import pandas as pd

# Build summary table
summary_data = {
    'Property': [
        'D(hot) [m²/s]',
        'D(cold) [m²/s]',
        'D(hot) / D(cold)',
        '',
        'K_S(hot) [H/(m³·Pa^0.5)]',
        'K_S(cold) [H/(m³·Pa^0.5)]',
        'K_S(cold) / K_S(hot)',
        '',
        'Steady-state flux [H/(m²·s)]',
        'Analytical flux [H/(m²·s)]',
        'Flux difference [%]',
        '',
        'c(x=0) [H/m³]',
        'c(x=L) [H/m³]',
        '',
        'Time to 95% steady state [s]',
    ],
    'Isothermal (750K)': [
        f"{D_isothermal:.4e}",
        f"{D_isothermal:.4e}",
        "1.0×",
        '',
        f"{K_S_iso:.4e}",
        f"{K_S_iso:.4e}",
        "1.0×",
        '',
        f"{flux_left_iso:.4e}",
        "–",
        "–",
        '',
        f"{c_isothermal[0]:.4e}",
        f"{c_isothermal[-1]:.4e}",
        '',
        f"{times_A[np.argmax(flux_A_norm >= 0.95)]:.1f}",
    ],
    'Gradient (900K → 600K)': [
        f"{D_hot:.4e}",
        f"{D_cold:.4e}",
        f"{D_hot/D_cold:.1f}×",
        '',
        f"{K_S_hot:.4e}",
        f"{K_S_cold:.4e}",
        f"{K_S_cold/K_S_hot:.1f}×",
        '',
        f"{flux_left_grad:.4e}",
        f"{J_analytical:.4e}",
        f"{100*abs(flux_left_grad - J_analytical)/J_analytical:.2f}%",
        '',
        f"{c_gradient[0]:.4e}",
        f"{c_gradient[-1]:.4e}",
        '',
        f"{times_B[np.argmax(flux_B_norm >= 0.95)]:.1f}",
    ]
}

df_summary = pd.DataFrame(summary_data)

print("\n" + "="*100)
print("SUMMARY TABLE: ISOTHERMAL VS. TEMPERATURE GRADIENT")
print("="*100)
print(df_summary.to_string(index=False))
print("="*100)

# Store for export
df_summary.to_csv('04_summary_table.csv', index=False)
print("\nTable saved to '04_summary_table.csv'")

## Key Physics Insights

### 1. **Temperature Gradient Breaks Linear Resistance**
In an isothermal slab, the concentration profile is linear (constant gradient). With a temperature gradient, the profile becomes **non-linear**: steeper on the cold side (where D is low) and shallower on the hot side (where D is high).

### 2. **Effective Permeance is NOT Simply Average**
The permeation flux does NOT equal $J = D_\text{avg} \times K_{S,\text{avg}} \times \Delta c$. Instead:

$$J = \frac{\Delta c}{\int_0^L \frac{dx}{D(T(x))}}$$

The **thermal resistance** $\int_0^L \frac{dx}{D(T(x))}$ is dominated by the coldest (low-D) region — a **harmonic mean** behavior.

### 3. **The Sievert Potential Remains Nearly Linear**
Although $c(x)$ and $K_S(x)$ both vary non-linearly, their ratio $\mu = c / K_S$ (the Sievert potential, or $\sqrt{P}$ equivalent) remains approximately linear. This is because $\mu$ satisfies:

$$\nabla \cdot (D(T) K_S(T) \nabla \mu) = 0$$

The product $D \times K_S$ varies less drastically than either alone.

### 4. **Faster Breakthrough with Higher Hot-Side Temperature**
The high D at the hot end allows tritium to enter the slab rapidly. This reduces the ramp-up time to steady state compared to the isothermal case, because the "bottleneck" (low D) is localized to the cold end, not distributed throughout the slab.

### 5. **Implications for Breeder Blanket Design**
- **Lower tritium inventory:** The gradient concentrates tritium near the cold side, reducing total stored tritium.
- **Higher extraction risk:** Hot-side permeation is rapid; cold-side extraction is slow (requires diffusion through low-D region).
- **Traps amplify the effect:** Temperature-dependent trap occupancy further increases the concentration gradient, with more trapping at the cold end.
- **Design implication:** Blanket designs must account for **spatially resolved** tritium distributions, not simple average concentrations.

## Next Steps: Notebook 05

In the next notebook, we will extend these concepts to **3D spherical geometry** (modeling a single pebble in a packed-bed breeder blanket) and see how:

1. **Radial temperature gradient** (hotter at outer surface, cooler at center) affects tritium transport
2. **Spherical geometry** changes the effective surface area and diffusion path
3. **Multiple pebbles** interacting through tritium exchange at solid-state contact points
4. **Time-dependent tritium release** from a pebble bed under transient heating

Let's build toward a full blanket-scale simulation!

---

### References and Further Reading

- **FESTIM 2.0 Documentation:** [festim.readthedocs.io](https://festim.readthedocs.io)  
- **Tritium breeding blankets:** Knitter, R. et al. (2009). *J. Nucl. Mater.*, 376(2), 68–80.  
- **Hydrogen in Li₂O:** Hollenberg, G. W. et al. (1994). *J. Nucl. Mater.*, 212–215, 1457–1463.  
- **Sievert's law and solubility:** ICONE 2020 symposium proceedings.
